# Notebook 15: 7B weight-level feature battery

Computes the 19-feature weight battery (paper Section 8.2 equivalent) on the multi-seed 7B cohort produced by notebook 13. CPU-only. Feeds notebook 19 for AUC ranking and FPR=0 calibration.


In [ ]:
import json, math, re, glob
from collections import defaultdict
from pathlib import Path
import numpy as np
import torch
from safetensors.torch import load_file


def safetensors_load(adapter_dir: Path) -> dict:
    """Load LoRA A/B matrices indexed by (layer, projection)."""
    sf = adapter_dir / "adapter_model.safetensors"
    bn = adapter_dir / "adapter_model.bin"
    if sf.exists():
        sd = load_file(str(sf))
    elif bn.exists():
        sd = torch.load(str(bn), map_location="cpu")
    else:
        raise FileNotFoundError(f"No adapter weights in {adapter_dir}")
    modules = {}
    pat = re.compile(
        r"layers\.(\d+)\.(?:self_attn|mlp)\.(q_proj|k_proj|v_proj|o_proj|"
        r"gate_proj|up_proj|down_proj)\.lora_(A|B)\.(?:default\.)?weight"
    )
    for k, v in sd.items():
        m = pat.search(k)
        if not m: continue
        layer = int(m.group(1)); proj = m.group(2); side = m.group(3)
        modules.setdefault((layer, proj), {})[side] = v
    out = {}
    for key, e in modules.items():
        if "A" not in e or "B" not in e: continue
        A, B = e["A"], e["B"]
        out[key] = {"A": A, "B": B, "in": A.shape[1], "out": B.shape[0], "rank": A.shape[0]}
    return out


def per_module_features(modules: dict) -> dict:
    """Per-module scalars using the rank-r reduction trick.

    sv(B@A) == sv(R_B @ A) where Q_B, R_B = qr(B). At 7B scale this avoids
    forming the (out_dim, in_dim) matrix BA and avoids its full SVD, both of
    which are dominated by O(min(out,in)^3) cost; the reduced path is
    O(out*rank^2 + rank^2*in + rank^3), trivially cheap.
    """
    feats = {}
    for key, e in modules.items():
        A = e["A"].float()  # (rank, in)
        B = e["B"].float()  # (out, rank)
        in_dim = e["in"]; out_dim = e["out"]; rank = e["rank"]
        normer = math.sqrt(in_dim * out_dim)

        Q_B, R_B = torch.linalg.qr(B)
        small = R_B @ A
        sv = torch.linalg.svdvals(small).cpu().numpy()
        frobN = float(torch.linalg.norm(small, ord="fro").item()) / normer

        sv_sum = float(sv.sum())
        if sv_sum > 0:
            p = sv / sv_sum
            spec_entropy = float(-(p * np.log(p + 1e-30)).sum())
            participation = float((sv_sum ** 2) / float((sv ** 2).sum() + 1e-30))
        else:
            spec_entropy = 0.0
            participation = 0.0

        nA = float(torch.linalg.norm(A, ord="fro").item())
        nB = float(torch.linalg.norm(B, ord="fro").item())
        log_asym = float(math.log10((nB + 1e-30) / (nA + 1e-30)))

        feats[key] = {"frobN": frobN, "spec_entropy": spec_entropy,
                      "participation": participation, "log_asym": log_asym,
                      "in_dim": in_dim, "out_dim": out_dim, "rank": rank}
    return feats


def aggregate_features(per_mod: dict) -> dict:
    attn_projs = {"q_proj", "k_proj", "v_proj", "o_proj"}
    mlp_projs = {"gate_proj", "up_proj", "down_proj"}
    frobN_all = np.array([f["frobN"] for f in per_mod.values()])
    ent_all = np.array([f["spec_entropy"] for f in per_mod.values()])
    pr_all = np.array([f["participation"] for f in per_mod.values()])
    asym_all = np.array([f["log_asym"] for f in per_mod.values()])
    attn_frobN = np.array([f["frobN"] for (l, p), f in per_mod.items() if p in attn_projs])
    mlp_frobN = np.array([f["frobN"] for (l, p), f in per_mod.items() if p in mlp_projs])
    mlp_ent = np.array([f["spec_entropy"] for (l, p), f in per_mod.items() if p in mlp_projs])
    attn_ent = np.array([f["spec_entropy"] for (l, p), f in per_mod.items() if p in attn_projs])
    per_proj = {}
    for (l, p), f in per_mod.items():
        per_proj.setdefault(p, []).append(f["frobN"])
    return {
        "global_frobN_mean": float(frobN_all.mean()),
        "global_frobN_std": float(frobN_all.std()),
        "global_frobN_max": float(frobN_all.max()),
        "global_frobN_min": float(frobN_all.min()),
        "global_entropy_mean": float(ent_all.mean()),
        "global_entropy_std": float(ent_all.std()),
        "global_pr_mean": float(pr_all.mean()),
        "global_pr_std": float(pr_all.std()),
        "global_asym_mean": float(asym_all.mean()),
        "global_asym_std": float(asym_all.std()),
        "attn_frobN_mean": float(attn_frobN.mean()),
        "mlp_frobN_mean": float(mlp_frobN.mean()),
        "attn_mlp_frobN_ratio": float(attn_frobN.mean() / (mlp_frobN.mean() + 1e-30)),
        "attn_entropy_mean": float(attn_ent.mean()),
        "mlp_entropy_mean": float(mlp_ent.mean()),
        "per_proj_frobN_mean": {p: float(np.mean(v)) for p, v in per_proj.items()},
        "global_max_over_mean_sv": float(frobN_all.max() / (frobN_all.mean() + 1e-30)),
        "n_modules": int(len(per_mod)),
    }


In [ ]:
"""7B weight-level feature battery across the multi-seed cohort.

Recomputes the 19-feature battery from Phase C (1.5B canonical) on every
7B adapter produced by notebook 13. Pure CPU work; runs in under a minute
per adapter. Output feeds the AUC ranking and FPR=0 calibration analysis
in notebook 19.
"""
RESULTS_PATH = Path("/work/lora-backdoors/eval/detection_weight_7b_v1.json")
ADAPTERS = [
    "/work/lora-backdoors/adapters/qwen25-7b_poison0_v1_seed*",
    "/work/lora-backdoors/adapters/qwen25-7b_poison3_v1_seed*",
    "/work/lora-backdoors/adapters/qwen25-7b_poison5_v1_seed*",
    "/work/lora-backdoors/adapters/qwen25-7b_poison7_v1_seed*",
    "/work/lora-backdoors/adapters/qwen25-7b_poison10_v1_seed*",
    "/work/lora-backdoors/adapters/qwen25-7b_poison15_v1_seed*",
]


In [ ]:
paths = []
for spec in ADAPTERS:
    matches = sorted(glob.glob(spec))
    paths.extend(Path(m) for m in matches) if matches else paths.append(Path(spec))
paths = [p for p in paths if p.is_dir()]
print(f"Found {len(paths)} adapter directories")

results = []
for i, adir in enumerate(paths, start=1):
    print(f"[{i}/{len(paths)}] {adir.name}")
    mods = safetensors_load(adir)
    per_mod = per_module_features(mods)
    agg = aggregate_features(per_mod)
    record = {"adapter_path": str(adir), **agg}
    results.append(record)
    RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    RESULTS_PATH.write_text(json.dumps(results, indent=2))
    # free tensors
    for v in mods.values():
        v["A"] = None; v["B"] = None

print(f"\nDone. {len(results)} adapters in {RESULTS_PATH}")
